# 12 – Human-in-the-Loop (HITL) & Auto-Ticket Node

The **auto_ticket_node** implements a two-pass HITL gate:
1. **Pass 1**: Detects anomalies → sets `pending_action` and pauses for approval
2. **Pass 2**: After `approved=True` → creates Jira tickets for each anomaly

This prevents runaway ticket creation without human confirmation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from graph.nodes import auto_ticket_node
from graph.state import initial_state

## 1. No anomalies — node is a no-op

In [ ]:
state = initial_state(query='show metrics')
result = auto_ticket_node(state)

print('pending_action:', result.get('pending_action'))
print('auto_tickets  :', result.get('auto_tickets'))
print('=> No anomalies, nothing to do')

## 2. Pass 1 — anomaly detected, approval requested

In [ ]:
state = initial_state(
    query='show retention metrics',
    anomalies=[
        'retention: GRR 78.0% is below threshold 85.0% — risk of missing targets',
        'retention: 87 at-risk accounts exceeds alert threshold 30',
    ],
    approved=False,
)
result = auto_ticket_node(state)

pending = result.get('pending_action')
print('pending_action type   :', pending.get('type'))
print('pending_action message:', pending.get('message'))
print('anomaly count         :', pending.get('count'))
print('products affected     :', pending.get('products'))
print('auto_tickets created  :', result.get('auto_tickets', []))

## 3. Pass 2 — approved, tickets created

In [ ]:
state_approved = initial_state(
    query='show retention metrics',
    anomalies=[
        'retention: GRR 78.0% is below threshold 85.0% — risk of missing targets',
        'ltv: LTV:CAC ratio 2.1 is below 3x minimum',
    ],
    approved=True,
)
result = auto_ticket_node(state_approved)

tickets = result.get('auto_tickets', [])
print(f'{len(tickets)} ticket(s) created:')
for t in tickets:
    fields = t.get('fields', {})
    print(f"  [{t.get('key')}] {fields.get('summary', '')[:60]}")
    print(f"    labels  : {fields.get('labels')}")
    print(f"    priority: {fields.get('priority', {}).get('name')}")

print('\npending_action after approval:', result.get('pending_action'))
print('approved reset to            :', result.get('approved'))

## 4. Full graph flow with anomaly → HITL → approval

In [ ]:
from graph.graph import build_graph
from services.databricks.mock import MockDatabricksService
from agents.information_agent import InformationAgent
from graph import nodes as _nodes

# Inject low-GRR data so anomalies are detected
_nodes._agents['information'] = InformationAgent(
    data_service=MockDatabricksService(low_grr=True)
)

graph = build_graph()

# Step 1: run query — should produce pending_action
state1 = initial_state(
    query='What are the retention metrics?',
    data_products=['retention'],
    approved=False,
)
r1 = graph.invoke(state1)

print('=== Step 1: Initial query ===')
print('Anomalies     :', r1.get('anomalies'))
print('Pending action:', r1.get('pending_action'))

if r1.get('pending_action'):
    # Step 2: approve
    print('\n=== Step 2: User approves ticket creation ===')
    state2 = {**r1, 'approved': True}
    r2 = auto_ticket_node(state2)
    tickets = r2.get('auto_tickets', [])
    print(f'Tickets created: {len(tickets)}')
    for t in tickets:
        print(f"  {t.get('key')}: {t.get('fields', {}).get('summary', '')[:50]}")
else:
    print('No anomalies detected (GRR may be normal)')

## 5. HITL keywords that trigger approval gate

In [ ]:
from graph.nodes import _HITL_KEYWORDS
print('HITL trigger keywords:', _HITL_KEYWORDS)

# Test which anomaly strings trigger the gate
test_anomalies = [
    'retention: GRR below threshold',
    'bookings: revenue missing for 3 days',
    'cac: model drift detected',
    'ltv: calculation complete',
]

for a in test_anomalies:
    triggers = any(kw in a.lower() for kw in _HITL_KEYWORDS)
    print(f'  {"TRIGGERS" if triggers else "skips  "} gate: {a}')